# World Cup 2026 — Match Outcome Predictor

**Goal:** predict international match outcomes — `home_win` / `draw` / `away_win` — for the 2026 World Cup.
**Approach:** a self-computed **Elo** from match history -> engineered features -> models
(logistic / gradient boosting / MLP), always split by **time** (never randomly).

## 1. Exploratory Data Analysis

**Data:** martj42 *International football results* (`results.csv`) — every international match
from 1872 to today, plus the scheduled 2026 World Cup fixtures.

> **Two datasets, one file.** Elo burns in over the **full history** (we keep every row), but we
> model — and therefore explore — only the **2010-onward training window**.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# Load the FULL history. Keep all of it: Elo "burns in" over every match ever played, so cutting
# to 2010 here would reset every team to 1500 in 2010 and wreck the early ratings.
df = pd.read_csv("results.csv", parse_dates=["date"])
print("Full history :", df.shape, "|", df.date.min().date(), "->", df.date.max().date())

# TRAINING WINDOW = 2010 onward. EDA + (later) model training use this slice, while the full
# `df` stays available to feed the Elo engine.
TRAIN_START = "2010-01-01"
df_train = df[df.date >= TRAIN_START].copy()
print("Training (>=2010):", df_train.shape,
      f"| burn-in rows kept for Elo: {len(df) - len(df_train):,}")
df_train.head()

Full history : (49363, 9) | 1872-11-30 -> 2026-06-27
Training (>=2010): (15776, 9) | burn-in rows kept for Elo: 33,587


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
33587,2010-01-02,Iran,North Korea,1.0,0.0,Friendly,Doha,Qatar,True
33588,2010-01-02,Qatar,Mali,0.0,0.0,Friendly,Doha,Qatar,False
33589,2010-01-02,Syria,Zimbabwe,6.0,0.0,Friendly,Kuala Lumpur,Malaysia,True
33590,2010-01-02,Yemen,Tajikistan,0.0,1.0,Friendly,Sana'a,Yemen,False
33591,2010-01-03,Angola,Gambia,1.0,1.0,Friendly,Vila Real de Santo António,Portugal,True


### Columns, dtypes & missing values *(training window)*

Within 2010+, only `home_score` / `away_score` are missing — those **72 rows are the unplayed
2026 World Cup fixtures**, not data-quality gaps.

In [2]:
print("Dtypes:")
print(df_train.dtypes.to_string())

missing = pd.DataFrame({
    "n_missing": df_train.isna().sum(),
    "pct_%": (df_train.isna().mean() * 100).round(2),
})
print("\nMissing values per column:")
print(missing.to_string())

Dtypes:
date          datetime64[ns]
home_team             object
away_team             object
home_score           float64
away_score           float64
tournament            object
city                  object
country               object
neutral                 bool

Missing values per column:
            n_missing  pct_%
date                0   0.00
home_team           0   0.00
away_team           0   0.00
home_score         72   0.46
away_score         72   0.46
tournament          0   0.00
city                0   0.00
country             0   0.00
neutral             0   0.00


### Goals & the home-advantage signal *(2010+)*

Home teams average clearly more goals than away teams even in the modern era — the raw signal
behind an explicit home-advantage term in the Elo.

In [3]:
print(df_train[["home_score", "away_score"]].describe().round(3).to_string())

home_mean, away_mean = df_train.home_score.mean(), df_train.away_score.mean()
print(f"\nMean goals  home: {home_mean:.3f}   away: {away_mean:.3f}")
print(f"Home edge:   +{home_mean - away_mean:.3f} goals/game")

       home_score  away_score
count   15704.000   15704.000
mean        1.609       1.119
std         1.643       1.359
min         0.000       0.000
25%         0.000       0.000
50%         1.000       1.000
75%         2.000       2.000
max        22.000      20.000

Mean goals  home: 1.609   away: 1.119
Home edge:   +0.491 goals/game


### Teams, tournaments & the `neutral` flag *(2010+)*

Fewer teams and tournaments than the full history (defunct nations drop out). Friendlies are
still a large share — noisy games we'll down-weight in the Elo K-factor.

In [4]:
print("Unique values:")
for c in ["home_team", "away_team", "tournament", "city", "country"]:
    print(f"  {c:<12} {df_train[c].nunique():>5}")

print("\nNeutral-venue flag:")
print(df_train.neutral.value_counts(dropna=False).to_string())

print("\nTop 12 tournaments by match count:")
print(df_train.tournament.value_counts().head(12).to_string())

Unique values:
  home_team      306
  away_team      301
  tournament     106
  city          1413
  country        225

Neutral-venue flag:
neutral
False    10948
True      4828

Top 12 tournaments by match count:
tournament
Friendly                                4972
FIFA World Cup qualification            3422
African Cup of Nations qualification    1052
UEFA Euro qualification                 1016
UEFA Nations League                      658
CONCACAF Nations League                  422
African Cup of Nations                   365
FIFA World Cup                           328
AFC Asian Cup qualification              269
Gold Cup                                 225
COSAFA Cup                               201
UEFA Euro                                184


### Outcome distribution — the label we predict

Computed on **played** 2010+ matches (the 72 future fixtures have no score yet). This is the
class balance the model must beat: home wins dominate, draws are the minority — and the hardest
class to call.

In [5]:
played = df_train.dropna(subset=["home_score", "away_score"]).copy()
played["result"] = np.select(
    [played.home_score > played.away_score, played.home_score < played.away_score],
    ["home_win", "away_win"], default="draw",
)
split = played.result.value_counts(normalize=True).mul(100).round(1)
print(f"Played matches since 2010: {len(played):,}\n")
print("Outcome split (%):")
print(split.to_string())

Played matches since 2010: 15,704

Outcome split (%):
result
home_win    47.8
away_win    29.0
draw        23.2


### Project-relevant checks

- **Training window** = 2010 onward; Elo still burns in over *all* history.
- The **72 missing-score rows** are future 2026 fixtures — Elo records their pre-match features
  but skips the rating update (leak-free by construction).
- No duplicate rows.

In [6]:
n_played = int(df_train[["home_score", "away_score"]].notna().all(axis=1).sum())
print("Rows in training window (>=2010):", len(df_train))
print("  of which played (have a score):", n_played)
print("  future 2026 fixtures (no score):", len(df_train) - n_played)
print("Burn-in rows kept (pre-2010, Elo only):", int((df.date < TRAIN_START).sum()))
print("Exact duplicate rows (full df):", int(df.duplicated().sum()))

Rows in training window (>=2010): 15776
  of which played (have a score): 15704
  future 2026 fixtures (no score): 72
Burn-in rows kept (pre-2010, Elo only): 33587
Exact duplicate rows (full df): 0
